In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "custseg").exists():   # project root = folder that holds the package
    if ROOT == ROOT.parent:
        raise RuntimeError("run the notebook from inside the project folder")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
from custseg import config, db
img = lambda p, w=900: display(Image(filename=str(ROOT / p), width=w))

# 02 · Data Quality
Results of the 26 PRD 4.3 checks (`custseg.validation`).

In [2]:
dq = pd.read_csv(ROOT / '05_data_engineering/validation/data_quality_results.csv')
dq[['check_id','prd_rule','layer','description','violations','total','violation_share','status','handling']]

,check_id,prd_rule,layer,description,violations,total,violation_share,status,handling
0,DQ01,Customer ID,raw,customer_id is null/empty,0,99441,0.000000,PASS,rows would be rejected in staging
1,DQ02,Customer ID,raw,customer_unique_id is null/empty,0,99441,0.000000,PASS,rows would be rejected in staging
2,DQ03,Transaction ID,raw,duplicated order_id in orders,0,99441,0.000000,PASS,DISTINCT ON (order_id) in staging
3,DQ04,Transaction ID,raw,"duplicated (order_id, order_item_id) line",0,112650,0.000000,PASS,DISTINCT ON in staging
4,DQ05,Transaction Date,staging,purchase timestamp missing / unparseable,0,99441,0.000000,PASS,cast fails loudly in staging; null -> dq_issue
5,DQ06,Transaction Date,staging,delivered before purchase (impossible lifecycle),0,99441,0.000000,PASS,delivery_days set NULL in fact_orders; revenue...
6,DQ07,Transaction Date,staging,approved before purchase,0,99441,0.000000,PASS,flag only (approval time not used in analysis)
7,DQ08,Transaction Date,staging,status 'delivered' without delivery date,8,96478,0.000083,PASS,kept as revenue (status is authoritative); exc...
8,DQ09,Quantity,warehouse,quantity <= 0 on a sales line,0,112650,0.000000,PASS,Olist: 1 line = 1 unit
9,DQ10,Unit Price,staging,negative or zero item price,0,112650,0.000000,PASS,would be rejected


In [3]:
dq.status.value_counts()

status
PASS    23
INFO     3
Name: count, dtype: int64

In [4]:
pd.read_csv(ROOT / '05_data_engineering/validation/feature_reconciliation.csv')

,feature,max_abs_diff,mismatches
0,recency_days,0.000000,0
1,frequency,0.000000,0
2,monetary,0.000000,0
3,avg_order_value,0.010000,0
4,tenure_days,0.000000,0
5,avg_items_per_order,0.000444,0
6,n_categories,0.000000,0
7,avg_installments,0.000333,0
8,voucher_order_share,0.000043,0
9,credit_card_order_share,0.000033,0


In [5]:
db.query("SELECT order_status, count(*) n, sum(is_revenue::int) revenue_orders FROM dw.fact_orders GROUP BY 1 ORDER BY 2 DESC")

,order_status,n,revenue_orders
0,delivered,96478,96211
1,shipped,1107,0
2,canceled,625,0
3,unavailable,609,0
4,invoiced,314,0
5,processing,301,0
6,created,5,0
7,approved,2,0
